In [ ]:
# Mount google drive to current runtime session to access required files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import Libraries
import shutil
import pandas as pd
import numpy as np
import os
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
import time

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
NVIDIA A100-SXM4-40GB


In [ ]:
# Import spectrogram zip to local storage for efficient loading

shutil.copy('/content/drive/MyDrive/DSAN6600 Project/piano_roll.zip',
            '/content/piano_roll.zip')

'/content/piano_roll.zip'

In [ ]:
# Unzip spectrogram file to extract all 150,000 npy files
!unzip -q piano_roll.zip -d /content/piano_roll

In [ ]:
# Size of spectrogram folder and number of npy files
!du -sh /content/piano_roll/piano_roll/*
!find /content/piano_roll/piano_roll/ -type f | wc -l

/bin/bash: line 1: /usr/bin/du: Argument list too long
150014


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DSAN6600 Project/lmd_full_metadata.csv')

# Filter out missing keys
df = df[df['key_name'].notna() & (df['key_name'] != '')]

# Filter major keys only
df = df[df['key_name'].str.contains('major')]

print(f"Total usable samples: {len(df)}")
print(f"Key distribution:\n{df['key_name'].value_counts()}")

Total usable samples: 93478
Key distribution:
key_name
C major     59720
G major      6836
F major      5821
D major      4520
A# major     3707
D# major     3560
A major      3025
E major      2291
G# major     1807
C# major     1050
B major       618
F# major      523
Name: count, dtype: int64


In [ ]:
# Cap C major at 4500
df_c = df[df['key_name'] == 'C major'].sample(4500, random_state=42)
df_others = df[df['key_name'] != 'C major']

# Combine and drop keys with less than 1000 samples
df_balanced = pd.concat([df_c, df_others])
df_balanced = df_balanced.groupby('key_name').filter(lambda x: len(x) >= 1000)

print(df_balanced['key_name'].value_counts())
print(f"Total samples: {len(df_balanced)}")
print(f"Total classes: {df_balanced['key_name'].nunique()}")

key_name
G major     6836
F major     5821
D major     4520
C major     4500
A# major    3707
D# major    3560
A major     3025
E major     2291
G# major    1807
C# major    1050
Name: count, dtype: int64
Total samples: 37117
Total classes: 10


In [ ]:
# Convert keys to integers for labeling purposes
key_to_idx = {key: idx for idx, key in enumerate(sorted(df_balanced['key_name'].unique()))}
idx_to_key = {idx: key for key, idx in key_to_idx.items()}

print(key_to_idx)
# {'A major': 0, 'A# major': 1, 'C major': 2, ...}

df_balanced['label'] = df_balanced['key_name'].map(key_to_idx)

df_balanced.head()

{'A major': 0, 'A# major': 1, 'C major': 2, 'C# major': 3, 'D major': 4, 'D# major': 5, 'E major': 6, 'F major': 7, 'G major': 8, 'G# major': 9}


,file,filepath,duration_s,resolution,n_instruments,n_notes,n_tempo_changes,initial_bpm,n_key_changes,initial_key,n_time_sig_changes,initial_time_sig,key_name,label
70182,2c877e367d9ecae70c59f9a1fafff7b6.mid,../data/raw_data/lmd_full/2/2c877e367d9ecae70c...,17.35,384,4,191,1,83.0,1,0.0,1,4/4,C major,2
147776,5c513934243de942f6b27b4383403f1f.mid,../data/raw_data/lmd_full/5/5c513934243de942f6...,203.91,480,15,6463,1,133.0,1,0.0,1,4/4,C major,2
132351,19684ed364581e059e7645042b607c86.mid,../data/raw_data/lmd_full/1/19684ed364581e059e...,359.91,240,11,5510,1,80.0,1,0.0,1,4/4,C major,2
88719,065ba7baee7cda0903cceb2c54404395.mid,../data/raw_data/lmd_full/0/065ba7baee7cda0903...,189.71,480,7,3059,2,70.0,1,0.0,1,4/4,C major,2
82806,ba68ba35a4bf1a58dc2619ac72509c8f.mid,../data/raw_data/lmd_full/b/ba68ba35a4bf1a58dc...,60.78,480,1,995,1,165.0,1,0.0,1,4/4,C major,2


In [ ]:
df_balanced['npy_file'] = df_balanced['file'].str.replace("mid","npy")

# Build full file paths
data_dir = '/content/piano_roll/piano_roll/'
df_balanced['file_path'] = data_dir + df_balanced['npy_file']

In [ ]:
sample = df_balanced['file_path'].iloc[1]
print(f"Example path: {sample}")
print(f"File exists: {os.path.exists(sample)}")

Example path: /content/piano_roll/piano_roll/5c513934243de942f6b27b4383403f1f.npy
File exists: True


In [ ]:
# Check which files actually exist
df_balanced['file_exists'] = df_balanced['file_path'].apply(os.path.exists)

print(f"Files found:   {df_balanced['file_exists'].sum()}")
print(f"Files missing: {(~df_balanced['file_exists']).sum()}")

# Keep only existing files
df_balanced = df_balanced[df_balanced['file_exists']]

df_balanced = df_balanced[df_balanced['file_path'].apply(
    lambda p: np.load(p).shape == (128, 468)
)]

print(f"Remaining samples after removing different shapes: {len(df_balanced)}")

NUM_CLASSES = df_balanced['key_name'].nunique()
print(f"Number of classes: {NUM_CLASSES}")

Files found:   36112
Files missing: 1005
Remaining samples after removing different shapes: 36112
Number of classes: 10


In [ ]:
sample = np.load('/content/piano_roll/piano_roll/deeb8ab7a04e5360231fb5019d847b1a.npy')
print(f"Shape: {sample.shape}")
print(f"Dtype: {sample.dtype}")
print(f"Min: {sample.min():.4f}, Max: {sample.max():.4f}")

Shape: (128, 468)
Dtype: uint8
Min: 0.0000, Max: 114.0000


In [ ]:
class PianoRollDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        x = np.load(self.file_paths[idx]).astype(np.float32)
        # Piano rolls are binary/velocity values in [0, 127] — normalize to [0, 1]
        x = x / 127.0
        x = x[np.newaxis, :, :]              # add channel dim → (1, 128, 468)
        return torch.tensor(x), torch.tensor(self.labels[idx])

In [ ]:
file_paths = df_balanced['file_path'].tolist()
labels = df_balanced['label'].tolist()

# Split into train/val

X_train, X_val, y_train, y_val = train_test_split(
    file_paths, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels  # ensures balanced split across classes
)

print(f"Train samples: {len(X_train)}")
print(f"Val samples:   {len(X_val)}")

train_dataset = PianoRollDataset(X_train, y_train)
val_dataset   = PianoRollDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True)

Train samples: 28889
Val samples:   7223


In [ ]:
CNN_CONFIGS = {
    'cnn3': [32, 64, 128],
    'cnn4': [32, 64, 128, 256],
    'cnn5': [32, 64, 128, 256, 512],
}

# GRU head hidden sizes for each depth option
GRU_HEAD_CONFIGS = {
    'head128': 128,   # original: gru_out → 128 → num_classes  (1 hidden layer)
    'head256': 256,   # wider head
    'head128x2': None # 2-layer head: gru_out → 256 → 128 → num_classes
}

def make_cnn_block(in_ch, out_ch):
    return [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
    ]

class HybridCNN(nn.Module):
    """
    Configurable CNN-GRU where depth/size is controlled by config strings.

    cnn_config:  'cnn3' | 'cnn4' (default) | 'cnn5'
    head_config: 'head128' (default, 1 hidden layer)
               | 'head256' (wider, 1 hidden layer)
               | 'head128x2' (2 hidden layers: 256→128)

    GRU input_size is always 2048 regardless of CNN depth (see comment above).
    gru_hidden: hidden size of the GRU (default 256)
    dropout:    applied to GRU between-layers and head input
    """
    def __init__(self, num_outputs, cnn_config='cnn4', head_config='head128',
                 gru_hidden=256, gru_layers=2, dropout=0.3):
        super().__init__()

        channels = CNN_CONFIGS[cnn_config]

        # Build CNN blocks dynamically
        layers = []
        in_ch = 1
        for out_ch in channels:
            layers += make_cnn_block(in_ch, out_ch)
            in_ch = out_ch
        self.features = nn.Sequential(*layers)

        # GRU
        self.gru = nn.GRU(
            input_size=2048,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0
        )

        gru_out_size = gru_hidden * 2  # bidirectional doubles output

        # Build head dynamically
        if head_config == 'head128':
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(gru_out_size, 128),
                nn.ReLU(),
                nn.Linear(128, num_outputs)
            )
        elif head_config == 'head256':
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(gru_out_size, 256),
                nn.ReLU(),
                nn.Linear(256, num_outputs)
            )
        elif head_config == 'head128x2':
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(gru_out_size, 256),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, num_outputs)
            )

    def forward(self, x):
        # --- CNN Spatial Extraction ---
        x = self.features(x)

        # --- Bridge to GRU ---
        B, C, F, T = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()  # (B, T, C, F)
        x = x.view(B, T, C * F)                  # (B, T, 2048)

        # --- GRU Temporal Modeling ---
        output, hidden = self.gru(x)

        # Extract final hidden state from BiGRU
        final_forward  = hidden[-2, :, :]
        final_backward = hidden[-1, :, :]
        final_state    = torch.cat((final_forward, final_backward), dim=1)

        # --- Output ---
        out = self.head(final_state)

        if out.shape[1] == 1:
            out = out.squeeze(1)

        return out

In [ ]:
class EarlyStopper:
    def __init__(self, patience=4, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_val_acc = 0

    def early_stop(self, validation_acc):
        if validation_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = validation_acc
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
# Default = original model configuration
DEFAULT_LR         = 3e-4
DEFAULT_DROPOUT    = 0.3
DEFAULT_GRU_HIDDEN = 256
DEFAULT_GRU_LAYERS = 2

learning_rates = [1e-3, 5e-4, 3e-4, 1e-4, 1e-5]
dropout_rates  = [0.2, 0.3, 0.4]

model_configs = [
    {'cnn_config': 'cnn3', 'head_config': 'head128',   'gru_hidden': DEFAULT_GRU_HIDDEN, 'gru_layers': DEFAULT_GRU_LAYERS},
    {'cnn_config': 'cnn4', 'head_config': 'head128',   'gru_hidden': DEFAULT_GRU_HIDDEN, 'gru_layers': DEFAULT_GRU_LAYERS},  # default
    {'cnn_config': 'cnn5', 'head_config': 'head256',   'gru_hidden': DEFAULT_GRU_HIDDEN, 'gru_layers': DEFAULT_GRU_LAYERS},
    {'cnn_config': 'cnn4', 'head_config': 'head128x2', 'gru_hidden': DEFAULT_GRU_HIDDEN, 'gru_layers': DEFAULT_GRU_LAYERS},
]

DEFAULT_MODEL = model_configs[1]  # cnn4, head128, gru_hidden=256, gru_layers=2

gru_configs = [
    {'gru_hidden': 128, 'gru_layers': 1},
    {'gru_hidden': 256, 'gru_layers': 1},
    {'gru_hidden': 256, 'gru_layers': 2},  # default
    {'gru_hidden': 512, 'gru_layers': 2},
    {'gru_hidden': 256, 'gru_layers': 3},
]

# Build 17 independent trials — one parameter varies, rest held at default
trials = (
    # Sweep 1: learning rate (5 trials)
    [{'lr': lr, 'dropout': DEFAULT_DROPOUT, **DEFAULT_MODEL}
     for lr in learning_rates]
    +
    # Sweep 2: dropout (3 trials)
    [{'lr': DEFAULT_LR, 'dropout': do, **DEFAULT_MODEL}
     for do in dropout_rates]
    +
    # Sweep 3: model depth/size (4 trials)
    [{'lr': DEFAULT_LR, 'dropout': DEFAULT_DROPOUT, **cfg}
     for cfg in model_configs]
    +
    # Sweep 4: GRU hidden size and layers (5 trials)
    [{'lr': DEFAULT_LR, 'dropout': DEFAULT_DROPOUT,
      'cnn_config': DEFAULT_MODEL['cnn_config'],
      'head_config': DEFAULT_MODEL['head_config'],
      **gru_cfg}
     for gru_cfg in gru_configs]
)

print(f"Total trials: {len(trials)}")  # 5 + 3 + 4 + 5 = 17

results   = []
OUTPUT_DIR = '/content/drive/MyDrive/cnn_model_output/hpsearch_cnngru/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for trial_num, params in enumerate(trials):
    lr          = params['lr']
    dropout     = params['dropout']
    cnn_config  = params['cnn_config']
    head_config = params['head_config']
    gru_hidden  = params['gru_hidden']
    gru_layers  = params['gru_layers']

    print(f"\n{'='*65}")
    print(f"Trial {trial_num+1}/{len(trials)} | lr={lr} | dropout={dropout} | "
          f"cnn={cnn_config} | head={head_config} | "
          f"gru_hidden={gru_hidden} | gru_layers={gru_layers}")
    print(f"{'='*65}")

    model = HybridCNN(
        num_outputs=NUM_CLASSES,
        cnn_config=cnn_config,
        head_config=head_config,
        gru_hidden=gru_hidden,
        gru_layers=gru_layers,
        dropout=dropout
    ).cuda()

    criterion     = nn.CrossEntropyLoss()
    optimizer     = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler        = GradScaler('cuda')
    scheduler     = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)
    early_stopper = EarlyStopper(patience=3, min_delta=0.25)

    best_val_acc = 0.0
    trial_start  = time.time()

    for epoch in range(15):
        # --- Training ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, lbls in train_loader:
            images, lbls = images.cuda(), lbls.cuda()
            optimizer.zero_grad()

            with autocast('cuda'):
                outputs = model(images)
                loss    = criterion(outputs, lbls)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            correct      += predicted.eq(lbls).sum().item()
            total        += lbls.size(0)

        train_acc = 100. * correct / total

        # --- Validation ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, lbls in val_loader:
                images, lbls = images.cuda(), lbls.cuda()
                outputs      = model(images)
                _, predicted  = outputs.max(1)
                val_correct  += predicted.eq(lbls).sum().item()
                val_total    += lbls.size(0)

        val_acc = 100. * val_correct / val_total

        # --- Save best model for this trial ---
        if val_acc > early_stopper.best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),
                       f'{OUTPUT_DIR}best_trial{trial_num+1}_{cnn_config}_{head_config}'
                       f'_gru{gru_hidden}x{gru_layers}_lr{lr}_do{dropout}.pth')

        print(f"  Epoch {epoch+1:2d}/15 | Loss: {running_loss/len(train_loader):.4f} | "
              f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")

        scheduler.step()

        if early_stopper.early_stop(val_acc):
            print(f"  Early stop at epoch {epoch+1}, best val: {early_stopper.best_val_acc:.1f}%")
            break

    trial_time = (time.time() - trial_start) / 60
    print(f"  Trial complete | Best val acc: {best_val_acc:.1f}% | Time: {trial_time:.1f} min")

    results.append({
        'trial':        trial_num + 1,
        'lr':           lr,
        'dropout':      dropout,
        'cnn_config':   cnn_config,
        'head_config':  head_config,
        'gru_hidden':   gru_hidden,
        'gru_layers':   gru_layers,
        'best_val_acc': best_val_acc,
        'time_min':     round(trial_time, 2),
    })

Total trials: 17

Trial 1/17 | lr=0.001 | dropout=0.3 | cnn=cnn4 | head=head128 | gru_hidden=256 | gru_layers=2
  Epoch  1/15 | Loss: 1.3828 | Train: 51.6% | Val: 72.6%
  Epoch  2/15 | Loss: 0.8558 | Train: 74.5% | Val: 75.3%
  Epoch  3/15 | Loss: 0.8085 | Train: 76.2% | Val: 68.4%
  Epoch  4/15 | Loss: 0.7817 | Train: 77.1% | Val: 76.2%
  Epoch  5/15 | Loss: 0.7561 | Train: 77.6% | Val: 75.4%
  Epoch  6/15 | Loss: 0.7120 | Train: 79.2% | Val: 76.0%
  Epoch  7/15 | Loss: 0.6883 | Train: 80.0% | Val: 78.2%
  Epoch  8/15 | Loss: 0.6696 | Train: 80.4% | Val: 76.9%
  Epoch  9/15 | Loss: 0.6527 | Train: 80.7% | Val: 78.9%
  Epoch 10/15 | Loss: 0.6331 | Train: 81.4% | Val: 78.7%
  Epoch 11/15 | Loss: 0.5763 | Train: 83.2% | Val: 78.6%
  Epoch 12/15 | Loss: 0.5516 | Train: 83.7% | Val: 79.1%
  Early stop at epoch 12, best val: 78.9%
  Trial complete | Best val acc: 79.1% | Time: 3.9 min

Trial 2/17 | lr=0.0005 | dropout=0.3 | cnn=cnn4 | head=head128 | gru_hidden=256 | gru_layers=2
  Epoch  1/

In [ ]:
results_df = pd.DataFrame(results).sort_values('best_val_acc', ascending=False)
print("\n" + "="*65)
print("HYPERPARAMETER SEARCH COMPLETE")
print("="*65)
print(results_df.to_string(index=False))

results_df.to_csv(f'{OUTPUT_DIR}hpsearch_cnngru_results.csv', index=False)
print(f"\nResults saved to {OUTPUT_DIR}hpsearch_cnngru_results.csv")

best = results_df.iloc[0]
print(f"\nBest config: lr={best['lr']} | dropout={best['dropout']} | "
      f"cnn={best['cnn_config']} | head={best['head_config']} | "
      f"gru_hidden={best['gru_hidden']} | gru_layers={best['gru_layers']} | "
      f"val_acc={best['best_val_acc']:.1f}%")


HYPERPARAMETER SEARCH COMPLETE
 trial      lr  dropout cnn_config head_config  gru_hidden  gru_layers  best_val_acc  time_min
    15 0.00030      0.3       cnn4     head128         256           2     80.783608      4.49
    13 0.00030      0.3       cnn4     head128         128           1     80.769763      4.56
     7 0.00030      0.3       cnn4     head128         256           2     80.755919      4.50
     3 0.00030      0.3       cnn4     head128         256           2     80.492870      4.76
    16 0.00030      0.3       cnn4     head128         512           2     80.174443      3.58
     8 0.00030      0.4       cnn4     head128         256           2     80.132909      3.82
     9 0.00030      0.3       cnn3     head128         256           2     80.119064      3.24
    17 0.00030      0.3       cnn4     head128         256           3     79.966773      4.59
     4 0.00010      0.3       cnn4     head128         256           2     79.482210      4.49
    10 0.00030    